# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshithavalli1006-spec/FlyRank--AI-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: Prioritize pages for refresh when they are stale, declining while still receiving demand, thin despite visibility, or showing page-one decay/low CTR.
Reason codes: stale_visible_page, declining_with_demand, thin_visible_page, page_one_decay_risk, low_ctr_visible_page, low_engagement_visible_page, and general_refresh_review.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
import os
os.makedirs("work/outputs", exist_ok=True)
print("Folder ready:", os.path.exists("work/outputs"))

Folder ready: True


In [2]:
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 299, done.
remote: Counting objects: 100% (193/193), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 299 (delta 130), reused 98 (delta 98), pack-reused 106 (from 1)
Receiving objects: 100% (299/299), 1.88 MiB | 4.21 MiB/s, done.
Resolving deltas: 100% (161/161), done.


In [8]:
import pandas as pd
import numpy as np

# Load the starter dataset
df = pd.read_csv("flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv")

def normalize(s):
    s = s.fillna(s.median())
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo) if hi > lo else s * 0

def percentile_rank(s):
    return s.rank(pct=True)

df["visibility_score"] = percentile_rank(np.log1p(df["impressions_90d"]))
df["freshness_risk_score"] = percentile_rank(df["days_since_last_update"])

df["position_opportunity_score"] = (
    (1 - normalize(df["avg_position"].clip(lower=1, upper=50)))
    * df["visibility_score"]
    * (df["avg_position"] > 0).astype(int)
)

df["depth_gap_score"] = (
    1 - percentile_rank(df["word_count"])
) * df["visibility_score"]

df["baseline_refresh_score"] = (
    0.40 * df["visibility_score"]
    + 0.30 * df["freshness_risk_score"]
    + 0.25 * df["position_opportunity_score"]
    + 0.05 * df["depth_gap_score"]
).clip(0, 1)

def reason_codes(row):
    reasons = []

    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")

    if row["trend_direction"].lower() == "down" and row["impressions_90d"] >= 100:
        reasons.append("declining_with_demand")

    if 0 < row["word_count"] < 1200 and row["impressions_90d"] >= 250:
        reasons.append("thin_visible_page")

    if 0 < row["avg_position"] <= 10 and row["content_age_days"] >= 180:
        reasons.append("page_one_decay_risk")

    if (
        row["impressions_90d"] >= 500
        and 0 < row["avg_position"] <= 20
        and row["ctr"] < 0.5
    ):
        reasons.append("low_ctr_visible_page")

    if row["sessions_90d"] >= 30 and (
        (0 < row["engagement_rate"] < 30)
        or (0 < row["scroll_rate"] < 30)
    ):
        reasons.append("low_engagement_visible_page")

    if not reasons:
        reasons.append("general_refresh_review")

    return "|".join(reasons)

df["reason_codes"] = df.apply(reason_codes, axis=1)

def suggested_action(row):
    reasons = set(row["reason_codes"].split("|"))

    if "thin_visible_page" in reasons:
        return "expand_and_refresh"
    if "low_ctr_visible_page" in reasons:
        return "refresh_and_review_ctr"
    if "stale_visible_page" in reasons or "declining_with_demand" in reasons:
        return "refresh"
    return "monitor"

df["suggested_action_baseline"] = df.apply(suggested_action, axis=1)

df["baseline_rank"] = (
    df["baseline_refresh_score"]
    .fillna(0)
    .rank(method="first", ascending=False)
    .astype(int)
)

# Write required output
output_path = "work/outputs/baseline_action_score.csv"
df.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(df))
print(df.nlargest(5, "baseline_refresh_score")[
    ["baseline_rank", "baseline_refresh_score",
     "reason_codes", "suggested_action_baseline"]
])


Saved: work/outputs/baseline_action_score.csv
Rows: 30000
       baseline_rank  baseline_refresh_score  \
10870              1                0.933579   
22197              2                0.928383   
16648              3                0.926606   
18803              4                0.923023   
20926              5                0.922196   

                                            reason_codes  \
10870   low_ctr_visible_page|low_engagement_visible_page   
22197  declining_with_demand|low_ctr_visible_page|low...   
16648  declining_with_demand|low_engagement_visible_page   
18803                        low_engagement_visible_page   
20926   low_ctr_visible_page|low_engagement_visible_page   

      suggested_action_baseline  
10870    refresh_and_review_ctr  
22197    refresh_and_review_ctr  
16648                   refresh  
18803                   monitor  
20926    refresh_and_review_ctr  


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
top20 = df.sort_values("baseline_refresh_score", ascending=False).head(20)

review_cols = [
    "baseline_rank",
    "baseline_refresh_score",
    "suggested_action_baseline",
    "reason_codes"
]

print(top20[review_cols].to_string(index=False))

 baseline_rank  baseline_refresh_score suggested_action_baseline                                                           reason_codes
             1                0.933579    refresh_and_review_ctr                       low_ctr_visible_page|low_engagement_visible_page
             2                0.928383    refresh_and_review_ctr declining_with_demand|low_ctr_visible_page|low_engagement_visible_page
             3                0.926606                   refresh                      declining_with_demand|low_engagement_visible_page
             4                0.923023                   monitor                                            low_engagement_visible_page
             5                0.922196    refresh_and_review_ctr                       low_ctr_visible_page|low_engagement_visible_page
             6                0.920755    refresh_and_review_ctr   page_one_decay_risk|low_ctr_visible_page|low_engagement_visible_page
             7                0.919993          

Top-20 review: The highest-ranked pages are prioritized because they have strong visibility combined with freshness risk, position opportunity, or content-depth gaps. The reason codes explain why each page was selected. Confidence is moderate because this is a transparent baseline rule rather than a trained predictive model. The picks could be wrong when the underlying data is missing, stale, or when a page is intentionally designed with short content or low engagement.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [10]:
print("Weak picks:")
print(
    df.nsmallest(5, "baseline_refresh_score")[
        ["baseline_rank", "baseline_refresh_score",
         "reason_codes", "suggested_action_baseline"]
    ].to_string(index=False)
)

print("\nLeakage check:")
print("Product flags excluded: YES")
print("Future-window features excluded: YES")
print("Label-derived trend fields used only for reason codes, not scoring: YES")


Weak picks:
 baseline_rank  baseline_refresh_score           reason_codes suggested_action_baseline
         22301                0.008023 general_refresh_review                   monitor
         22300                0.008108 general_refresh_review                   monitor
         22299                0.008188 general_refresh_review                   monitor
         22298                0.008370 general_refresh_review                   monitor
         22297                0.008404 general_refresh_review                   monitor

Leakage check:
Product flags excluded: YES
Future-window features excluded: YES
Label-derived trend fields used only for reason codes, not scoring: YES


Weak picks + leakage check: The lowest-ranked pages have fewer signals indicating an immediate refresh opportunity, so they are reasonable weak picks. The baseline score does not use product flags or future-window features. Trend information is used only to explain the baseline recommendation and is not included in the numeric score.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.